# 02 · Protocolo de validación, métricas y auditoría de fugas (secciones 2 y 4)

- `src/backtest.py`: `backtest(forecast_fn, panel)` con origen móvil diario (92 orígenes), reentrenamiento mensual y adaptadores para PyTorch y Nixtla.
- `src/metrics.py`: MAE, RMSE, WAPE, MAPE, MASE, pinball y cobertura, con desgloses por cliente, por paso del horizonte, por hora del día y para la semana del 22 al 28 de diciembre.
- `src/train.py`: loop de entrenamiento con early stopping sobre validación y registro de cada iteración.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import torch
from src import data as D, metrics as M, backtest as B
from src.dataset import WindowDataset

panel = D.build_panel()
scale = M.mase_scale(panel)          # escala del MASE: seasonal naive 24 h en entrenamiento
print(len(panel["unique_id"].unique()), "clientes")

## 1. Contrato de `backtest()`

```python
forecast_fn(hist, origin, H) -> DataFrame(unique_id, ds, y_hat [, q10, q50, q90])
```

En cada origen (00:00 de cada día de prueba) `hist` contiene **solo** filas con `ds < origin`, es decir hasta las 23:00 del día anterior. El backtest verifica que cada pronóstico cubra exactamente las 24 horas `[origin, origin + 23]` para todos los clientes.

Ejemplo con el seasonal naive de 24 h (repetir el día anterior):

In [ ]:
def seasonal_naive_24(hist, origin, H):
    ids, x = B.tail_matrix(hist, 24)
    return B.frame_from_matrix(ids, origin, x[:, -24:, 0][:, :H])

fc = B.backtest(seasonal_naive_24, panel, name="SeasonalNaive24")
fc.head()

### Validación contra StatsForecast

El mismo modelo con `StatsForecast.cross_validation(h=24, n_windows=92, step_size=24)` debe dar exactamente los mismos pronósticos. En Nixtla `cutoff` es la última hora observada, así que `origin = cutoff + 1h`.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

sf = StatsForecast(models=[SeasonalNaive(season_length=24)], freq="h", n_jobs=-1)
cv = sf.cross_validation(df=panel[["unique_id", "ds", "y"]], h=24, n_windows=92, step_size=24)
fc_sf = B.from_nixtla_cv(cv, "SeasonalNaive")

comp = fc.merge(fc_sf, on=["unique_id", "ds", "origin"], suffixes=("", "_sf"))
assert len(comp) == len(fc) == len(fc_sf)
print("orígenes:", fc_sf["origin"].min().date(), "→", fc_sf["origin"].max().date(), f"({fc_sf['origin'].nunique()})")
print("diferencia máxima:", float((comp["y_hat"] - comp["y_hat_sf"]).abs().max()))

## 2. Métricas (en kWh)

In [ ]:
ev = M.evaluate(fc, panel, scale)
pd.Series(ev["resumen"]).to_frame("SeasonalNaive24")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
ev["por_h"]["MAE"].plot(ax=axes[0], marker="o", ms=3, title="MAE por paso del horizonte")
axes[0].set_xlabel("h (horas después del origen)"); axes[0].set_ylabel("MAE (kWh)")
ev["por_hora"]["MAE"].plot(ax=axes[1], marker="o", ms=3, title="MAE por hora del día", color="C1")
axes[1].set_xlabel("Hora del día")
for a in axes: a.grid(alpha=.3)
fig.tight_layout()

Con un origen a las 00:00, el paso `h` y la hora del día coinciden (h = hora + 1); las dos curvas se separan solo si algún modelo usa otra hora de origen.

**MAPE y clientes con consumo cercano a cero.** El MAPE divide cada error entre el valor real; cuando el real es ~0 el cociente explota (o se indefine si es exactamente 0). Aquí se calcula solo sobre horas con consumo > 0.001 kWh y se reporta cuántas se excluyeron: un cliente que pasa horas apagado puede dominar el MAPE con errores de pocos kWh. Por eso la comparación principal usa **MASE** (escala propia de cada cliente) y **WAPE** (suma de errores entre suma de consumo, dominado por los clientes grandes).

In [ ]:
print("Horas excluidas del MAPE por consumo ~0:", ev["resumen"]["MAPE_excluidas"])
pc = ev["por_cliente"].sort_values("MASE")
pd.concat([pc.head(3), pc.tail(3)])

In [ ]:
print("Semana del 22 al 28 de diciembre:")
pd.Series(ev["navidad"]).to_frame("SeasonalNaive24")

## 3. Auditoría de fugas (las dos restantes)

### Split aleatorio de ventanas

Si se mezclan las ventanas al azar y se reparten en train/val, dos ventanas de la misma serie separadas por una hora comparten 167 de sus 168 horas de entrada: el modelo valida sobre datos que prácticamente vio. Nuestro pipeline asigna cada ventana a un split **por fecha** (todas sus horas objetivo deben caer en el bloque), así que train, validación y prueba quedan ordenados en el tiempo.

In [ ]:
z = D.SeriesScaler().fit(panel).transform(panel)
splits = {s: WindowDataset(z, split=s) for s in ("train", "val", "test")}

def target_range(ds):
    first = ds.times[ds.origin_idx.min()]
    last = ds.times[ds.origin_idx.max() + ds.H - 1]
    return first, last

for s, w in splits.items():
    print(f"{s:5s}: objetivos de {target_range(w)[0]} a {target_range(w)[1]}")
assert target_range(splits["train"])[1] < target_range(splits["val"])[0]
assert target_range(splits["val"])[1] < target_range(splits["test"])[0]
print("OK: los bloques no se solapan y están en orden temporal")

In [ ]:
# Contraejemplo: lo que pasaría con un split aleatorio (como torch.utils.data.random_split)
todo = WindowDataset(z, split=None)
rng = np.random.default_rng(0)
perm = rng.permutation(len(todo))
val_idx, train_idx = perm[: len(perm) // 10], perm[len(perm) // 10 :]

# Marca, por serie y hora, qué orígenes quedaron en entrenamiento
S, T_ = len(todo.ids), len(todo.times)
en_train = np.zeros((S, T_), dtype=bool)
en_train[todo.series_idx[train_idx], todo.origin_idx[train_idx]] = True

# Una ventana de validación comparte horas objetivo con una de entrenamiento si hay un
# origen de entrenamiento de la misma serie a menos de H = 24 h de distancia
s_v, t_v = todo.series_idx[val_idx], todo.origin_idx[val_idx]
comparte = np.zeros(len(val_idx), dtype=bool)
for k in range(-23, 24):
    t = np.clip(t_v + k, 0, T_ - 1)
    comparte |= en_train[s_v, t]
print(f"Split aleatorio: {comparte.mean():.1%} de las ventanas de validación comparten horas objetivo "
      "con alguna ventana de entrenamiento de la misma serie (y casi todas sus 168 horas de entrada).")

### Early stopping con la prueba

`train.fit()` recibe un `val_ds` y antes de entrenar verifica que **ninguna hora objetivo de validación** sea igual o posterior a la primera hora que se va a pronosticar (`forecast_start`, por defecto 2014-10-01). La prueba solo la ve `backtest()`, una vez por modelo, con los pesos ya fijos. Si alguien pasa la prueba como validación, `fit` se detiene con un error:

In [ ]:
from torch import nn
from src import train as T

class Lineal(nn.Module):
    def __init__(self, L=168, H=24):
        super().__init__(); self.f = nn.Linear(L, H)
    def forward(self, x):
        return self.f(x[..., 0])

scaler = D.SeriesScaler().fit(panel)
val_mal = WindowDataset(z, split="test", origin_hour=0)
try:
    T.fit(Lineal(), splits["train"], val_mal, scaler, scale, name="no_deberia_correr", epochs=1, verbose=False)
    raise RuntimeError("fit aceptó la prueba como validación")
except AssertionError as e:
    print("OK, fit se niega:", e)

## 4. Ejemplo de punta a punta: modelo propio + `fit` + `backtest`

Un modelo lineal mínimo (168 → 24) para comprobar que todas las piezas encajan. La validación usa los orígenes de las 00:00, igual que el protocolo de prueba.

In [ ]:
train_ds = splits["train"]
val_ds = WindowDataset(z, split="val", origin_hour=0)
res = T.fit(Lineal(), train_ds, val_ds, scaler, scale, name="ejemplo_lineal",
            epochs=5, max_batches_per_epoch=400, batch_size=1024)
res.history

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fc_lin = B.backtest(B.torch_forecast_fn(res.model, scaler, L=168, device=device), panel, name="ejemplo_lineal")
pd.DataFrame({"SeasonalNaive24": M.summary_row("SeasonalNaive24", ev),
              "ejemplo_lineal": M.summary_row("ejemplo_lineal", M.evaluate(fc_lin, panel, scale))}).T

## 5. Reentrenamiento mensual (ventana expandible)

`backtest_retrain(fit_fn, panel)` llama a `fit_fn(datos con ds < inicio de mes)` el 1 de octubre, 1 de noviembre y 1 de diciembre, y evalúa cada mes con su modelo. Con el seasonal naive no cambia nada (no tiene parámetros); sirve para validar la mecánica antes de usarlo con el mejor modelo y el mejor baseline.

In [ ]:
fc_rt = B.backtest_retrain(lambda train_panel: seasonal_naive_24, panel, name="SeasonalNaive24_retrain")
assert fc_rt["origin"].nunique() == 92
assert np.allclose(fc_rt.sort_values(["unique_id", "ds"])["y_hat"].to_numpy(),
                   fc.sort_values(["unique_id", "ds"])["y_hat"].to_numpy())
print("OK: 92 orígenes, mismos pronósticos que sin reentrenar")